In [ ]:
import torch


# Activation Functions

**Mathematics:**

- **ReLU (Rectified Linear Unit):**
    
    $$f(x) = \max(0, x)$$
    
    $$f'(x) = \begin{cases} 1 & \text{if } x > 0 \\ 0 & \text{if } x \leq 0 \end{cases}$$
    
- **Leaky ReLU:** Fixes the "dying ReLU" problem where neurons get stuck outputting 0.
    
    $$f(x) = \max(\alpha x, x)$$


In [ ]:
import numpy as np
# ReLU
class ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        # return inputs where inputs > 0, else 0
        return np.maximun(0, inputs)

    def backward(self, dvalues):
        # dvalues is the gradient coming from the next layer
        dinputs = dvalues.copy()
        # setting gradients as 0 where gradient < 0
        dinputs[self.inputs <= 0] = 0
        return dinputs

# leaky ReLU
class leakyReLU:
    def __init__(self, alpha):
        self.alpha = 0.01

    def forward(self, inputs):
        self.inputs = inputs
        # return inputs where inputs > 0, else alpha * inputs
        return np.maximum(self.alpha * inputs, inputs)

    def backward(self, dvalues):
        dinputs = dvalues.copy()
        # If input <= 0, multiply the incoming gradient by alpha
        dinputs[self.inputs <= 0] = self.alpha * self.inputs

#### **Softmax**

Used for multi-class classification in the output layer. It turns raw scores (logits) into probabilities.

$$S(x_i) = \frac{e^{x_i}}{\sum_{j} e^{x_j}}$$

**Numerical Stability:**

Calculating $e^{x}$ directly can result in overflow (getting `inf`) if $x$ is large. A common trick is to subtract the maximum value from the inputs before exponentiation:

$$\frac{e^{x_i - C}}{\sum e^{x_j - C}}$$

where $C = \max(x)$. This shifts the range to $(-\infty, 0]$, ensuring stability.

In [ ]:
class Softmax:
    def forward(self, inputs):
        self.inputs = inputs
        #
        exp_values = np.exp(inputs - np.max(inputs, axis = 1, keepdims = True))

        #normalize them for each sample
        probabilities = exp_values / np.sum(exp_values, axis = 1, keepdims= True)

        self.output = probabilities
        return probabilities
    
    def backward(self, dvalues):
        # Create empty array for gradients as placeholder 
        self.dinputs = np.empty_like(dvalues)

        for index, (single_output, single_dvalue) in enumerate(zip(self.output, dvalues)):
            # flatten output array
            single_output = single_output.reshape(-1, 1)

            # calculate jacobian matrix of the softmax
            # formula: diag(S) - Dot(S, S_transpose)
            jacobian_matrix = np.diagflat(single_output) - np.dot(single_output, single_output.T)

            # calculate sample wise gradient and add it to the array of sample gradient
            # This effectively sums the gradients across the Jacobian
            self.dinputs[index] = np.dot(jacobian_matrix, single_dvalue)

        return self.dinputs

#### The Legacy Option: Sigmoid

You generally shouldn't use this for hidden layers anymore, but it is still standard for **Binary Classification** output (0 vs 1).

**The Math:**

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

**The Derivative:**

The beauty of Sigmoid is its clean derivative expressed in terms of itself:

$$\sigma'(x) = \sigma(x) \cdot (1 - \sigma(x))$$

In [ ]:
class Sigmoid:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = 1 / 1 + np.exp(-inputs)
        return self.output
    
    def backward(self, dvalues):
        # Derivative: sigmoid * (1 - sigmoid)
        # multiply by incoming gradient (chain rule)
        self.dinputs = dvalues * (self.output * (1 - self.output))
        return self.dinputs